- Codex : https://openai.com/ko-KR/codex/ 윈도우 설치 다운로드 실행
- 윈도우 검색창 >> Chat GPT
- 프롬프트 
    내 컴퓨터 바탕화면에 사과폴더 만들어줘
- 프롬프트 
    1. 내가 첨부한 05_Agent.ipynb를 파악해줘
    2. 위의 코드를 기반으로 아래의 기능을 추가하여 발전한 파이썬 코드를 제작해줘
        - 2-1) GUI 기능 추가(수신이메일/앱비밀번호 사용자 세팅 & 체크박스를 통해서 저장할 수 있게)
        - 2-2) D:\Edu\Machine Learning\workspace\20.업무자동화\codex결과.ipynb 파일 작성 후 저장해줘

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import pandas as pd
import os

from concurrent.futures import ThreadPoolExecutor, as_completed


# 대기시간
st = 3


# 검색 키워드
keywords = [
    "에어컨",
    "선풍기",
    "제습기",
    "냉풍기",
    "서큘레이터"
]



# ==========================================
# 하나의 키워드 검색 함수
# ==========================================

def naver_cafe_search(keyword):

    result = []


    print("="*60)
    print(f"{keyword} 검색 시작")
    print("="*60)



    # 크롬 실행
    driver = webdriver.Chrome()


    try:

        # 네이버 접속
        driver.get("https://www.naver.com")
        time.sleep(st)



        # 검색창
        green_box_ele = driver.find_element(
            By.ID,
            "query"
        )


        # 검색어 입력
        green_box_ele.send_keys(keyword)



        # 검색 버튼 클릭
        ai_s_else = driver.find_element(
            By.CLASS_NAME,
            "ai_effect_symbol"
        )

        ai_s_else.click()

        time.sleep(st)



        # 카페 메뉴 클릭
        elements = driver.find_elements(
            By.CLASS_NAME,
            "wxxES_QvNoYHFNCZ"
        )


        for element in elements:

            if element.text.strip() == "카페":

                element.click()

                print(
                    f"{keyword} 카페 클릭 완료"
                )

                break


        time.sleep(st)



        # 게시글 가져오기
        title_links = driver.find_elements(
            By.CLASS_NAME,
            "title_link"
        )



        for element in title_links:


            title = element.text.strip()

            url = element.get_attribute(
                "href"
            )


            print(
                keyword,
                title
            )


            result.append(
                {
                    "키워드": keyword,
                    "게시글제목": title,
                    "URL": url
                }
            )


    except Exception as e:

        print(
            keyword,
            "오류 발생 :",
            e
        )


    finally:

        # 브라우저 종료
        driver.quit()



    print(
        f"{keyword} 검색 완료"
    )


    return result





# ==========================================
# 병렬 실행
# ==========================================


total_result = []



# 동시에 실행할 브라우저 개수
with ThreadPoolExecutor(max_workers=5) as executor:


    futures = []


    for keyword in keywords:

        futures.append(
            executor.submit(
                naver_cafe_search,
                keyword
            )
        )



    # 결과 수집
    for future in as_completed(futures):

        data = future.result()

        total_result.extend(data)




# ==========================================
# Excel 저장
# ==========================================


df = pd.DataFrame(
    total_result
)



save_path = r"D:\Edu\Machine Learning\workspace\20.업무자동화\결과\웹"



if not os.path.exists(save_path):

    os.makedirs(save_path)



file_path = os.path.join(
    save_path,
    "naver_cafe_elect02.xlsx"
)



df.to_excel(
    file_path,
    index=False
)



print("="*70)
print("Excel 저장 완료")
print(file_path)
print("="*70)

print(df)

In [2]:
"""네이버 카페 검색 결과를 Excel로 저장하고 이메일로 보내는 GUI 프로그램.

필수 패키지: pip install selenium pandas openpyxl keyring
Chrome 및 ChromeDriver(Selenium Manager가 자동 설치할 수 있음)가 필요합니다.
"""
from __future__ import annotations

import json
import os
import queue
import re
import smtplib
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from email.message import EmailMessage
from pathlib import Path
from tkinter import BooleanVar, StringVar, Tk, messagebox, ttk
from tkinter.scrolledtext import ScrolledText

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

try:
    import keyring
except ImportError:
    keyring = None


APP_NAME = "NaverCafeAgent"
CONFIG_PATH = Path.home() / ".naver_cafe_agent_settings.json"
PASSWORD_SERVICE = "NaverCafeAgentGmail"
DEFAULT_KEYWORDS = "에어컨, 선풍기, 제습기, 냉풍기, 서큘레이터"

def search_naver_cafe(keyword: str, wait_seconds: int, headless: bool) -> list[dict[str, str]]:
    """한 키워드의 네이버 카페 검색 결과를 수집한다."""
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-notifications")
    options.add_argument("--window-size=1440,1000")
    driver = webdriver.Chrome(options=options)
    result: list[dict[str, str]] = []
    try:
        driver.get("https://search.naver.com/search.naver?where=view&query=" + keyword)
        wait = WebDriverWait(driver, max(wait_seconds, 3))
        # 네이버 화면 변경에 대비하여 현재 노출되는 결과 링크를 수집합니다.
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a.title_link")))
        for element in driver.find_elements(By.CSS_SELECTOR, "a.title_link"):
            title = element.text.strip()
            url = element.get_attribute("href")
            if title and url:
                result.append({"키워드": keyword, "게시글제목": title, "URL": url})
    finally:
        driver.quit()
    return result


def send_result_email(sender: str, app_password: str, recipient: str, attachment: Path) -> None:
    """Gmail SMTP로 생성된 Excel 파일을 첨부해 발송한다."""
    message = EmailMessage()
    message["Subject"] = f"네이버 카페 검색 결과 - {datetime.now():%Y-%m-%d %H:%M}"
    message["From"] = sender
    message["To"] = recipient
    message.set_content("네이버 카페 검색 결과 Excel 파일을 첨부합니다.")
    message.add_attachment(
        attachment.read_bytes(),
        maintype="application",
        subtype="vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        filename=attachment.name,
    )
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=30) as smtp:
        smtp.login(sender, app_password.replace(" ", ""))
        smtp.send_message(message)


class AgentApp:
    def __init__(self, root: Tk) -> None:
        self.root = root
        self.root.title("네이버 카페 검색 Agent")
        self.root.geometry("720x620")
        self.events: queue.Queue[tuple[str, object]] = queue.Queue()

        self.sender_var = StringVar()
        self.recipient_var = StringVar()
        self.password_var = StringVar()
        self.keywords_var = StringVar(value=DEFAULT_KEYWORDS)
        self.output_var = StringVar(value=str(Path.home() / "Documents" / "naver_cafe_results"))
        self.save_settings_var = BooleanVar(value=True)
        self.headless_var = BooleanVar(value=False)
        self._build_ui()
        self._load_settings()
        self.root.after(150, self._process_events)

    def _build_ui(self) -> None:
        frame = ttk.Frame(self.root, padding=16)
        frame.pack(fill="both", expand=True)
        frame.columnconfigure(1, weight=1)

        fields = [
            ("발신 Gmail", self.sender_var, False),
            ("수신 이메일", self.recipient_var, False),
            ("Gmail 앱 비밀번호", self.password_var, True),
            ("검색 키워드 (, 구분)", self.keywords_var, False),
            ("결과 저장 폴더", self.output_var, False),
        ]
        for row, (label, variable, secret) in enumerate(fields):
            ttk.Label(frame, text=label).grid(row=row, column=0, sticky="w", pady=5, padx=(0, 12))
            ttk.Entry(frame, textvariable=variable, show="*" if secret else "").grid(
                row=row, column=1, sticky="ew", pady=5
            )

        ttk.Checkbutton(
            frame,
            text="이 설정을 저장하기 (앱 비밀번호는 Windows 자격 증명 저장소에 안전하게 보관)",
            variable=self.save_settings_var,
        ).grid(row=5, column=0, columnspan=2, sticky="w", pady=(8, 2))
        ttk.Checkbutton(frame, text="브라우저 창 숨기기", variable=self.headless_var).grid(
            row=6, column=0, columnspan=2, sticky="w", pady=2
        )
        self.run_button = ttk.Button(frame, text="검색 · Excel 저장 · 이메일 발송", command=self._start)
        self.run_button.grid(row=7, column=0, columnspan=2, sticky="ew", pady=(12, 8))
        self.log = ScrolledText(frame, height=18, state="disabled")
        self.log.grid(row=8, column=0, columnspan=2, sticky="nsew")
        frame.rowconfigure(8, weight=1)

    def _load_settings(self) -> None:
        if not CONFIG_PATH.exists():
            return
        try:
            data = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
            self.sender_var.set(data.get("sender", ""))
            self.recipient_var.set(data.get("recipient", ""))
            self.keywords_var.set(data.get("keywords", DEFAULT_KEYWORDS))
            self.output_var.set(data.get("output_dir", self.output_var.get()))
            self.headless_var.set(data.get("headless", False))
            if keyring and self.sender_var.get():
                self.password_var.set(keyring.get_password(PASSWORD_SERVICE, self.sender_var.get()) or "")
        except (OSError, json.JSONDecodeError) as error:
            self._write_log(f"저장 설정을 불러오지 못했습니다: {error}")

    def _save_settings(self) -> None:
        if not self.save_settings_var.get():
            return
        data = {
            "sender": self.sender_var.get().strip(),
            "recipient": self.recipient_var.get().strip(),
            "keywords": self.keywords_var.get().strip(),
            "output_dir": self.output_var.get().strip(),
            "headless": self.headless_var.get(),
        }
        CONFIG_PATH.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
        if keyring:
            keyring.set_password(PASSWORD_SERVICE, data["sender"], self.password_var.get())
        else:
            self._write_log("keyring 미설치: 앱 비밀번호는 저장하지 않았습니다. pip install keyring 후 다시 실행하세요.")

    def _start(self) -> None:
        sender, recipient, password = self.sender_var.get().strip(), self.recipient_var.get().strip(), self.password_var.get().strip()
        keywords = [item.strip() for item in self.keywords_var.get().split(",") if item.strip()]
        if not all((sender, recipient, password, keywords)):
            messagebox.showwarning("입력 확인", "발신 Gmail, 수신 이메일, 앱 비밀번호, 검색 키워드를 모두 입력하세요.")
            return
        if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", sender) or not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", recipient):
            messagebox.showwarning("이메일 확인", "올바른 발신 Gmail 및 수신 이메일 주소를 입력하세요.")
            return
        self._save_settings()
        self.run_button.config(state="disabled")
        self._write_log("작업을 시작합니다.")
        threading.Thread(target=self._run_agent, args=(keywords,), daemon=True).start()

    def _run_agent(self, keywords: list[str]) -> None:
        try:
            total: list[dict[str, str]] = []
            workers = min(len(keywords), 5)
            with ThreadPoolExecutor(max_workers=workers) as executor:
                futures = {executor.submit(search_naver_cafe, word, 10, self.headless_var.get()): word for word in keywords}
                for future in as_completed(futures):
                    word = futures[future]
                    data = future.result()
                    total.extend(data)
                    self.events.put(("log", f"{word}: {len(data)}건 수집 완료"))
            output_dir = Path(self.output_var.get()).expanduser()
            output_dir.mkdir(parents=True, exist_ok=True)
            file_path = output_dir / f"naver_cafe_{datetime.now():%Y%m%d_%H%M%S}.xlsx"
            pd.DataFrame(total, columns=["키워드", "게시글제목", "URL"]).to_excel(file_path, index=False)
            self.events.put(("log", f"Excel 저장 완료: {file_path}"))
            send_result_email(self.sender_var.get().strip(), self.password_var.get().strip(), self.recipient_var.get().strip(), file_path)
            self.events.put(("done", f"완료되었습니다. {len(total)}건을 이메일로 발송했습니다.\n{file_path}"))
        except Exception as error:
            self.events.put(("error", f"작업 중 오류가 발생했습니다:\n{error}"))

    def _process_events(self) -> None:
        while not self.events.empty():
            kind, payload = self.events.get()
            if kind == "log":
                self._write_log(str(payload))
            elif kind == "done":
                self._write_log(str(payload))
                self.run_button.config(state="normal")
                messagebox.showinfo("완료", str(payload))
            elif kind == "error":
                self._write_log(str(payload))
                self.run_button.config(state="normal")
                messagebox.showerror("오류", str(payload))
        self.root.after(150, self._process_events)

    def _write_log(self, text: str) -> None:
        self.log.config(state="normal")
        self.log.insert("end", text + "\n")
        self.log.see("end")
        self.log.config(state="disabled")


if __name__ == "__main__":
    app_root = Tk()
    AgentApp(app_root)
    app_root.mainloop()

In [ ]:
# End of -------------------------------------------------